<a href="https://colab.research.google.com/github/vaishnavikabbe/AIML/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vaishnavikabbe/AIML/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [6]:
# ML-08 setup
# Load the AIML repository and starter dataset safely.

import os
import sys
import subprocess
import pandas as pd
import numpy as np

REPO_DIR = "/content/AIML"

# Clone your repository if it is not already in Colab
if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", "https://github.com/vaishnavikabbe/AIML.git", REPO_DIR],
        check=True
    )

os.chdir(REPO_DIR)

DATA_PATH = "data/raw/content_refresh_anonymized.csv"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. "
        "Make sure the AIML repository contains the starter CSV."
    )

df = pd.read_csv(DATA_PATH)

print("Repository:", os.getcwd())
print("Dataset loaded successfully.")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Repository: /content/AIML
Dataset loaded successfully.
Rows: 30000
Columns: 44


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Method choice

I will use a Random Forest classifier because the task is to identify pages that are observed as declining or not declining. Random Forest can capture non-linear relationships between search-performance and content signals without requiring a simple linear relationship.

It also provides feature importance values that can help explain which measured signals the model relies on. I will use the model as decision-support for prioritizing pages for review, not as causal proof or a prediction of Google's ranking algorithm.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 1: Confirm the target

if "trend_direction" not in df.columns:
    raise KeyError("trend_direction column not found.")

# Official task definition:
# declining = trend_direction == "down"

df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

print("Target distribution:")
print(df["is_declining_label"].value_counts())

print("\nTarget proportions:")
print(df["is_declining_label"].value_counts(normalize=True).round(3))

Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Target proportions:
is_declining_label
1    0.542
0    0.458
Name: proportion, dtype: float64


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### Split design

I will use a client-level holdout split. Pages from the same client should not appear in both the training and test sets because pages belonging to the same client may share patterns.

Keeping clients separate makes the evaluation more honest: the model is tested on clients it did not see during training.

The split will use approximately 80% of the clients for training and 20% for testing.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 2: Client-level split

# Find the client column used by the dataset.
possible_client_columns = [
    "client_id",
    "client",
    "client_key",
    "site_id"
]

client_col = None

for col in possible_client_columns:
    if col in df.columns:
        client_col = col
        break

if client_col is None:
    raise KeyError(
        "Could not find a client identifier column. "
        f"Available columns are: {list(df.columns)}"
    )

print("Client column:", client_col)
print("Unique clients:", df[client_col].nunique())

rng = np.random.RandomState(42)

clients = df[client_col].dropna().unique()
rng.shuffle(clients)

split_point = int(len(clients) * 0.80)

train_clients = set(clients[:split_point])
test_clients = set(clients[split_point:])

train_df = df[df[client_col].isin(train_clients)].copy()
test_df = df[df[client_col].isin(test_clients)].copy()

print("\nTrain rows:", len(train_df))
print("Test rows:", len(test_df))

print("\nTrain clients:", train_df[client_col].nunique())
print("Test clients:", test_df[client_col].nunique())

overlap = (
    set(train_df[client_col])
    & set(test_df[client_col])
)

print("\nClient overlap:", len(overlap))

assert len(overlap) == 0, "Client leakage detected!"

print("Client-level split verified.")

Client column: client_id
Unique clients: 32

Train rows: 22389
Test rows: 7611

Train clients: 25
Test clients: 7

Client overlap: 0
Client-level split verified.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

### Train and compare

The main evaluation metric is Precision@50. This measures how many of the top 50 pages selected by the system are actually observed as declining pages.

This metric matches the business action because the content team has limited review capacity and needs a short list of pages to inspect first.

I will compare the learned Random Forest model with a simple baseline ranking using 90-day impressions. Both will be evaluated on the same held-out clients.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 3: Train Random Forest and compare with baseline

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import precision_score

# --------------------------------------------------
# Feature selection
# --------------------------------------------------

# Numeric features that are available before prediction.
candidate_features = [
    "search_volume",
    "impressions_90d",
    "word_count"
]

# Add other safe numeric columns automatically,
# while explicitly excluding label/future/leakage fields.
excluded_keywords = [
    "trend",
    "label",
    "future",
    "product",
    "target"
]

numeric_columns = df.select_dtypes(
    include=[np.number]
).columns.tolist()

safe_numeric = []

for col in numeric_columns:
    if col == "is_declining_label":
        continue

    if any(
        keyword in col.lower()
        for keyword in excluded_keywords
    ):
        continue

    safe_numeric.append(col)

# Use available safe numeric features.
feature_columns = list(dict.fromkeys(
    candidate_features + safe_numeric
))

feature_columns = [
    col for col in feature_columns
    if col in df.columns
]

print("Features used:")
print(feature_columns)

# --------------------------------------------------
# Train/test data
# --------------------------------------------------

X_train = train_df[feature_columns].copy()
X_test = test_df[feature_columns].copy()

y_train = train_df["is_declining_label"]
y_test = test_df["is_declining_label"]

# --------------------------------------------------
# Model
# --------------------------------------------------

model = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "classifier",
        RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            class_weight="balanced",
            n_jobs=-1
        )
    )
])

model.fit(X_train, y_train)

# Predicted probability of declining
model_probability = model.predict_proba(X_test)[:, 1]

# --------------------------------------------------
# Precision@50 helper
# --------------------------------------------------

def precision_at_k(y_true, scores, k=50):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    k = min(k, len(y_true))

    top_indices = np.argsort(scores)[::-1][:k]

    return y_true[top_indices].mean()

# --------------------------------------------------
# Model Precision@50
# --------------------------------------------------

model_p50 = precision_at_k(
    y_test.values,
    model_probability,
    50
)

# --------------------------------------------------
# Baseline
# --------------------------------------------------
# Lower impressions = higher review priority.

baseline_score = -test_df["impressions_90d"].fillna(
    train_df["impressions_90d"].median()
).values

baseline_p50 = precision_at_k(
    y_test.values,
    baseline_score,
    50
)

# --------------------------------------------------
# Results table
# --------------------------------------------------

results = pd.DataFrame({
    "Method": [
        "Baseline: low impressions",
        "Random Forest"
    ],
    "Precision@50": [
        baseline_p50,
        model_p50
    ]
})

print("\nModel comparison:")
print(results.to_string(index=False))

print(
    "\nRandom Forest improvement over baseline:",
    round(model_p50 - baseline_p50, 3)
)

Features used:
['search_volume', 'impressions_90d', 'word_count', 'competition', 'cpc', 'char_count', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier_order', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']

Model comparison:
                   Method  Precision@50
Baseline: low impressions           0.1
            Random Forest           1.0

Random Forest improvement over baseline: 0.9


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

### Errors and interpretation

I will inspect the top-ranked pages where the model is most confident and compare them with their observed labels. False positives are pages that the model ranks highly but are not labeled as declining, while false negatives are declining pages that receive lower scores.

The model should be interpreted as learning patterns associated with the observed declining label. A high feature importance does not mean that the feature causes decline. The results are directional decision-support evidence for content review.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4: Error analysis and feature importance

# --------------------------------------------------
# Add predictions to test data
# --------------------------------------------------

error_df = test_df[
    [
        client_col,
        "is_declining_label",
        "search_volume",
        "impressions_90d",
        "word_count"
    ]
].copy()

error_df["model_probability"] = model_probability

error_df["predicted_label"] = (
    error_df["model_probability"] >= 0.5
).astype(int)

error_df["error_type"] = "Correct"

error_df.loc[
    (error_df["predicted_label"] == 1) &
    (error_df["is_declining_label"] == 0),
    "error_type"
] = "False positive"

error_df.loc[
    (error_df["predicted_label"] == 0) &
    (error_df["is_declining_label"] == 1),
    "error_type"
] = "False negative"

print("Error counts:")
print(error_df["error_type"].value_counts())

# --------------------------------------------------
# Show highest-confidence false positives
# --------------------------------------------------

false_positives = error_df[
    error_df["error_type"] == "False positive"
].sort_values(
    "model_probability",
    ascending=False
)

print("\nTop false positives:")
print(
    false_positives.head(10).to_string(index=False)
)

# --------------------------------------------------
# Show false negatives
# --------------------------------------------------

false_negatives = error_df[
    error_df["error_type"] == "False negative"
].sort_values(
    "model_probability",
    ascending=False
)

print("\nExample false negatives:")
print(
    false_negatives.head(10).to_string(index=False)
)

# --------------------------------------------------
# Feature importance
# --------------------------------------------------

rf = model.named_steps["classifier"]

importance = pd.DataFrame({
    "feature": feature_columns,
    "importance": rf.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print("\nTop feature importances:")
print(importance.head(10).to_string(index=False))

Error counts:
error_type
Correct           6395
False positive    1096
False negative     120
Name: count, dtype: int64

Top false positives:
        client_id  is_declining_label  search_volume  impressions_90d  word_count  model_probability  predicted_label     error_type
client_6208ef0f77                   0            0.0             2789      5717.0              0.840                1 False positive
client_6208ef0f77                   0            0.0             1664      3146.0              0.815                1 False positive
client_349c41201b                   0            0.0              677      3912.0              0.800                1 False positive
client_6208ef0f77                   0            0.0             1653      5382.0              0.795                1 False positive
client_6208ef0f77                   0            0.0              360      4233.0              0.780                1 False positive
client_6208ef0f77                   0            0.0        

### Final interpretation

The Random Forest provides a learned ranking that can be compared with the simple baseline using Precision@50. If the model achieves higher Precision@50, it provides evidence that combining multiple measured signals can improve the prioritization queue compared with the baseline rule.

The errors show that some pages can still be incorrectly prioritized or missed. Therefore, the model should support human review rather than automatically deciding which pages must be changed.

The feature importance results describe which available signals the model relied on most strongly. They do not establish that those signals cause changes in search performance.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.